In [1]:
!pip install transformers torch

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [4]:
def generate_reply(chat_history, user_message):

    # Add user message
    chat_history.append({
        "role": "user",
        "content": user_message
    })

    # Convert to model format
    prompt = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate response
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    # Decode only new tokens
    reply = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    # Save response
    chat_history.append({
        "role": "assistant",
        "content": reply
    })

    return chat_history, reply

In [5]:
def run_chatbot():

    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

    chat_history = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant."
        }
    ]

    while True:

        user_input = input("User: ")

        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye!")
            break

        chat_history, reply = generate_reply(chat_history, user_input)

        print("Chatbot:", reply)

In [6]:
run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?
User: Hello
Chatbot: Hello! How can I help you today? Is there something specific you would like to know or discuss? I'm here to answer any questions you might have.
User: What is Artificial Intelligence?
Chatbot: Artificial intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans do. It involves creating algorithms, software, and systems capable of performing tasks that typically require human intelligence such as speech recognition, image processing, decision-making, natural language understanding, and problem-solving.

There are several types of AI:

1. **Weak AI** (Narrow AI): This type of AI focuses on performing specific tasks. Examples include voice assistants, recommendation engines, and self-driving cars.
  
2. **Strong AI**: Also known as artificial general intelligence (AGI), this aims to create an intelligent machine with capabilities simi